# Distances

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

DATA = "data/stl10_binary"
IMG = 96

def load_rgb(split):
    X = np.fromfile(f"{DATA}/{split}_X.bin", dtype=np.uint8).reshape(-1, 3, IMG, IMG)
    return X.transpose(0, 3, 2, 1)

d = np.load("data/embeddings_stl10_1000.npz")

X = np.concatenate([load_rgb("train"), load_rgb("test")])[d["idx"]]
y = d["y"]
class_names = [l.strip() for l in open(f"{DATA}/class_names.txt") if l.strip()]

print(X.shape, y.shape)

## measures

In [ ]:
def cosine(E, i, j):
    return E[i] @ E[j] / (np.linalg.norm(E[i]) * np.linalg.norm(E[j]))

def euclidean(E, i, j):
    return np.linalg.norm(E[i] - E[j])

def dot(E, i, j):
    return E[i] @ E[j]

MEASURES = {"cosine": cosine, "euclidean": euclidean, "dot": dot}

## cosine, euclidean, dot

In [ ]:
c = y[3]
a, b = np.where(y == c)[0][:2]
pairs = {"same class": (a, b), "different": (6, 3)}

for name, f in MEASURES.items():
    print(name)
    for layer in (3, 12):
        E = d[f"E_l{layer}"]
        print(f"  layer {layer:2}", {k: round(float(f(E, *p)), 2) for k, p in pairs.items()})

fig, axes = plt.subplots(2, 2, figsize=(4.4, 4.8))

for row, (tag, (i, j)) in enumerate(pairs.items()):
    for col, k in enumerate((i, j)):
        axes[row, col].imshow(X[k])
        axes[row, col].set_title(f"{tag}\n{class_names[y[k] - 1]} ({k})", fontsize=8)
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

## top 5 similar / unsimilar

In [ ]:
LAYER = 12
q = 6

E = d[f"E_l{LAYER}"]
sims = (E @ E[q]) / (np.linalg.norm(E, axis=1) * np.linalg.norm(E[q]))

others = np.arange(len(E)) != q
order = np.where(others)[0][np.argsort(sims[others])]

unsimilar = order[:5]
similar = order[-5:][::-1]

for tag, idx in (("similar", similar), ("unsimilar", unsimilar)):
    print(tag, [f"{class_names[y[k] - 1]} {sims[k]:+.2f}" for k in idx])

In [ ]:
fig, axes = plt.subplots(1, 12, figsize=(18, 2.8),
                         gridspec_kw={"width_ratios": [1.6] + [1] * 5 + [0.45] + [1] * 5})

axes[0].imshow(X[q])
axes[0].set_title(f"query\n{class_names[y[q] - 1]}", fontsize=8)
axes[0].axis("off")

axes[6].axis("off")

for ax, k in zip(np.r_[axes[1:6], axes[7:]], np.r_[similar, unsimilar]):
    ax.imshow(X[k])
    ax.set_title(f"{class_names[y[k] - 1]}\n{sims[k]:+.2f}", fontsize=7)
    ax.axis("off")

plt.tight_layout(rect=(0, 0, 1, 0.88))

mid = lambda a: (a[0].get_position().x0 + a[-1].get_position().x1) / 2
fig.text(0.5, 0.975, "cosine similarity", ha="center", fontsize=11)
fig.text(mid(axes[1:6]), 0.90, "most similar", ha="center", fontsize=9)
fig.text(mid(axes[7:]), 0.90, "most unsimilar", ha="center", fontsize=9)

plt.show()